# Set Up

**Token-augmented dual autoencoder.** This notebook is a variant of `autoencoder.ipynb` that adds a third, cell-level histology branch built from ViT *token* embeddings (16x16 sub-tiles per 224px patch), alongside the existing cell-level transcriptomics branch and patch-level histology branch. See `autoencoder_functions_new.py` (`DualAutoencoder`'s `token_dim` docstring) for the added machinery -- it's purely additive, so `autoencoder.ipynb` is unaffected and still the current baseline.

Kept as a **separate notebook** (rather than folded into `autoencoder.ipynb`) so the two can be run side by side without one crowding the other, and so a with/without-tokens comparison is just "run this notebook with `CONFIG["use_tokens"]` flipped" rather than editing cells back and forth.

**Before running this notebook**, per-slide token data must exist at `data/{slide}/01_preprocessing/unimodel/metadata/cell_token_embeddings.h5ad` -- produced by `token_preprocessing.py`:
```
python token_preprocessing.py --slide CRC_P1
```
This distills the raw ~9GB/slide token grid down to two compact per-cell vectors (`footprint`: tokens under the nucleus; `halo`: tokens in a small ring around it), so the notebook itself never touches the raw per-token files. Currently only `CRC_P1` has been processed -- `CONFIG["sample"]` below defaults to `"CRC_P1"` accordingly. Once more slides are processed, this can move to `"combined"` to match `autoencoder.ipynb`'s scope directly.

In [1]:
# import
import anndata as ad
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import scanpy as sc
import scipy.sparse as sp
import squidpy as sq
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import SAGEConv
from torch_geometric.nn import TransformerConv

%load_ext autoreload
%autoreload 2

# Same shared module as autoencoder.ipynb -- the token-branch additions live in
# here (guarded by token_dim=None / use_tokens checks), nothing was duplicated.
from autoencoder_functions_new import *


/home/charlottewolf/miniconda3/envs/autoencoder/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
CONFIG = dict(
    sample="combined",       # str -> single slide. list[str] or "combined" -> multi-slide.
                            # all 5 slides in the combined transcriptomics adata now have
                            # `cell_token_embeddings.h5ad` (CRC_P1, CRC_P2, CRC_P5, Normal_P3,
                            # Normal_P5), matching autoencoder.ipynb's scope.
    bin_size="002um",
    dir_cellseg="cell_seg_stardist",  # in case of multiple cell segmentation methods
    loss_fct="ntxent",                # optional: ntxent, cosine

    # model
    latent_dim=64,                      # latent dim for encoders
    proj_dim=32,                        # dimension of projection head for contrastive learning
    pool_type="attention",              # "attention" or "mean" -> pooling type for pooling cells to patch level
    encoder="transconv",                # sage or transconv for transcriptomics encoding

    # graph / batching
    n_neighs=6,                         # how many neighbors in knn spatial graph of cells
    num_neighbors=(10, 10),
    patches_per_batch=64,
    max_cells_per_patch=64,

    # loss
    temperature=0.07,
    neighbor_radius=2912,     # 2x patch diameter, currently 224x224 px

    # false-negative protection: exclude transcriptomically near-duplicate,
    # non-matching patches/cells from the hard-negative set (cosine-similarity
    # cutoff on the trans-side projection), same rationale as neighbor_radius
    # but gated on expression similarity instead of spatial distance -- see
    # `patch_level_infonce`'s docstring. NEW, not yet empirically validated on
    # this dataset -- check retrieval accuracy with/without before trusting the
    # default threshold. Set to None to disable (reproduces prior behavior).
    patch_anchor_sim_threshold=0.9,
    local_anchor_sim_threshold=0.9,

    weight_reconstruction_by_patch_freq=True,

    hist_batch_correct=False,
    hist_harmony_npcs=50,

    hist_pretrain_epochs=200,
    hist_pretrain_lr=1e-3,
    hist_pretrain_weight_decay=1e-5,
    hist_pretrain_val_frac=0.1,
    hist_pretrain_patience=15,

    # training
    n_epochs=100,
    lr=1e-3,
    warmup_epochs=5,
    max_contrast_weight=0.0,

    # --- token branch (this notebook only) ---
    use_tokens=True,                    # master toggle -- False reproduces autoencoder.ipynb's
                                         # behavior exactly (DualAutoencoder gets token_dim=None).
    token_pretrain_epochs=100,
    token_pretrain_lr=1e-3,
    token_pretrain_weight_decay=1e-5,
    token_pretrain_val_frac=0.1,
    token_pretrain_patience=15,
    token_lr_mult=1.0,
    freeze_token=True,

    local_neighbor_radius=50,           # px -- cell-level analogue of neighbor_radius (2912 for
                                         # patches). Median nearest-cell spacing measured ~27px in
                                         # this dataset, so this excludes only genuinely adjacent
                                         # cells from the negative set, not the whole neighborhood.
    max_local_contrast_weight=0.5,
    local_warmup_epochs=5,
)


In [6]:
basedir = Path("../../data")
dir_histology = basedir / "combined/01_preprocessing/unimodel/metadata"
dir_transcriptomics = basedir / "combined/01_preprocessing" / CONFIG["dir_cellseg"] / CONFIG["bin_size"] / "custom_binning"
_run_tag = CONFIG["sample"] if isinstance(CONFIG["sample"], str) else "combined"
_variant_tag = "with_tokens" if CONFIG["use_tokens"] else "no_tokens"
dir_output = basedir / _run_tag / "03_integration/autoencoders" / CONFIG["bin_size"] / _variant_tag
dir_output.mkdir(parents=True, exist_ok=True)
print("Writing outputs to:", dir_output)


Writing outputs to: ../../data/combined/03_integration/autoencoders/002um/with_tokens


# Running the Pipeline

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -- load transcriptomics + patch histology (identical to autoencoder.ipynb) --
f_hist_adata = sc.read_h5ad(dir_histology / "filtered_WSI_patch_embeddings.h5ad")
f_trans_adata = ad.read_h5ad(dir_transcriptomics / "count_area_filtered_adata.h5ad")
f_trans_coord = gpd.read_parquet(dir_transcriptomics / "nuclei_geometry.parquet")

if CONFIG["sample"] == "combined":
    pass
elif isinstance(CONFIG["sample"], str):
    f_hist_adata = f_hist_adata[f_hist_adata.obs["Slide_ID"] == CONFIG["sample"]]
    f_trans_adata = f_trans_adata[f_trans_adata.obs["Slide_ID"] == CONFIG["sample"]]
    f_trans_coord = f_trans_coord[f_trans_coord["Slide_ID"] == CONFIG["sample"]]
elif isinstance(CONFIG["sample"], (list, tuple)):
    f_hist_adata = f_hist_adata[f_hist_adata.obs["Slide_ID"].isin(CONFIG["sample"])]
    f_trans_adata = f_trans_adata[f_trans_adata.obs["Slide_ID"].isin(CONFIG["sample"])]
    f_trans_coord = f_trans_coord[f_trans_coord["Slide_ID"].isin(CONFIG["sample"])]
print("Training on slides:", sorted(f_trans_adata.obs["Slide_ID"].unique().tolist()))

trans_global_id = f_trans_adata.obs["Slide_ID"].astype(str) + "_" + f_trans_adata.obs["id"].astype(str)
if not trans_global_id.is_unique:
    raise ValueError("Cannot set Slide_ID+id as index: contains duplicates")
f_trans_adata.obs = f_trans_adata.obs.set_index(trans_global_id)

hist_global_id = f_hist_adata.obs["Slide_ID"].astype(str) + "_" + f_hist_adata.obs["Patch_ID"].astype(str)
if not hist_global_id.is_unique:
    raise ValueError("Cannot set Slide_ID+Patch_ID as index: contains duplicates")
f_hist_adata.obs = f_hist_adata.obs.set_index(hist_global_id)


/home/charlottewolf/miniconda3/envs/autoencoder/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


Training on slides: ['CRC_P1', 'CRC_P2', 'CRC_P5', 'Normal_P3', 'Normal_P5']


### Load token data (separate cell -- skipped entirely when `use_tokens=False`)

Loads each in-scope slide's `cell_token_embeddings.h5ad` (from `token_preprocessing.py`), concatenates them, and -- if some in-scope slides don't have token data yet -- restricts training to the slides that do (rather than silently zero-filling, which would let the model learn "zero vector = slide X" as a shortcut).

In [ ]:
# Note: we run this once per slide ? 
if CONFIG["use_tokens"]:
    slides_needed = sorted(f_trans_adata.obs["Slide_ID"].astype(str).unique())
    token_adatas, slides_with_tokens = [], []
    for s in slides_needed:
        p = basedir / s / "01_preprocessing/unimodel/metadata/cell_token_embeddings.h5ad"
        if p.exists():
            token_adatas.append(sc.read_h5ad(p))
            slides_with_tokens.append(s)
        else:
            print(f"WARNING: no token data for slide {s} ({p}) -- run token_preprocessing.py for it first")

    missing = sorted(set(slides_needed) - set(slides_with_tokens))
    if missing:
        print(f"Restricting to slides with token data: {slides_with_tokens} (dropping {missing})")
        f_hist_adata = f_hist_adata[f_hist_adata.obs["Slide_ID"].isin(slides_with_tokens)].copy()
        f_trans_adata = f_trans_adata[f_trans_adata.obs["Slide_ID"].astype(str).isin(slides_with_tokens)].copy()
        f_trans_coord = f_trans_coord[f_trans_coord["Slide_ID"].isin(slides_with_tokens)].copy()
    if not token_adatas:
        raise RuntimeError("use_tokens=True but no in-scope slide has token data -- run token_preprocessing.py first")

    f_token_adata = ad.concat(token_adatas, join="outer")
    token_key = f_token_adata.obs["Slide_ID"].astype(str) + "_" + f_token_adata.obs["id"].astype(str)
    if not token_key.is_unique:
        raise ValueError("Cannot set Slide_ID+id as token index: contains duplicates")
    f_token_adata.obs = f_token_adata.obs.set_index(token_key)
    f_token_adata = f_token_adata[f_token_adata.obs["matched"].astype(bool)].copy()
    print(f"Loaded token data: {f_token_adata.n_obs} cells across {len(slides_with_tokens)} slide(s)")
else:
    f_token_adata = None


In [ ]:
if CONFIG["hist_batch_correct"]:
    f_hist_adata.obsm["X_harmony"] = harmonize_hist_embeddings(
        f_hist_adata, batch_key="Slide_ID", n_pcs=CONFIG["hist_harmony_npcs"]
    )


### Data prep

Same steps as `autoencoder.ipynb` (SVG mask, spatial graph, cell<->patch pairing, standardization, patch metadata, PyG data + loader), plus -- guarded by `use_tokens` -- aligning each surviving cell to its token footprint/halo vectors and building the cell-level metadata (`cell_meta`) the local contrastive term needs.

In [ ]:
# 1. SVG mask (reduced gene expression vector with only SVGs kept)
print("1. SVG mask")
svg_mask = get_svg_union_mask(f_trans_adata)
print(f"{svg_mask.sum()} SVGs selected (union)")

# 2. spatial graph - build spatial graph: nodes = cells, contain gene exp vector
print("2. building spatial graph")
conn = build_spatial_graph(f_trans_adata, library_key="Slide_ID", n_neighs=CONFIG["n_neighs"], delaunay=False)

# 3. pairing -> get pairing of uni hist embedding to cell SVGs
print("3. pairing")
keep_mask, patch_row_trans, hist_emb = build_pairing(f_trans_adata, f_hist_adata)

# 3b. token pairing: further restrict keep_mask to cells that also have token data
if CONFIG["use_tokens"]:
    has_token = f_trans_adata.obs_names.isin(f_token_adata.obs_names)
    n_dropped = int(keep_mask.sum() - (keep_mask & has_token.values).sum())
    if n_dropped:
        print(f"  dropping {n_dropped} cells with a hist-patch match but no token match")
    keep_mask = keep_mask & has_token.values

trans_sub = f_trans_adata[keep_mask].copy()
conn_sub = conn[keep_mask][:, keep_mask]
X_svg = trans_sub.X[:, svg_mask]
X_svg = X_svg.toarray() if sp.issparse(X_svg) else np.asarray(X_svg)

slide_codes, slide_uniques = pd.factorize(trans_sub.obs["Slide_ID"])
n_slides = len(slide_uniques)

# 4. standardize
print("4. standardising")
X_svg, hist_emb, svg_scaler, hist_scaler = standardize_modalities(X_svg, hist_emb)

if CONFIG["use_tokens"]:
    token_sub = f_token_adata[trans_sub.obs_names]
    assert (token_sub.obs_names == trans_sub.obs_names).all(), "token/trans row alignment mismatch"
    # footprint + halo, concatenated -- see conversation notes (TRIPLEX's target/neighbor
    # split): kept as two distinct signals rather than blended into one average.
    token_emb = np.concatenate(
        [np.asarray(token_sub.obsm["X_token_footprint"]), np.asarray(token_sub.obsm["X_token_halo"])],
        axis=1,
    ).astype(np.float32)
    token_scaler = StandardScaler()
    token_emb = token_scaler.fit_transform(token_emb).astype(np.float32)
    print(f"  token_emb: {token_emb.shape}")

# 5. patch metadata: centroid, cell count, inverse-frequency weight
print("5. centroids")
cell_xy = extract_cell_xy(trans_sub, f_trans_coord)
patch_meta, patch_meta_df = compute_patch_metadata(
    patch_row_trans, cell_xy, f_hist_adata,
    neighbor_radius=CONFIG["neighbor_radius"],
)
patch_meta_df.to_csv(dir_output / "patch_metadata.csv", index=False)

if CONFIG["use_tokens"]:
    cell_meta = build_cell_meta(cell_xy, slide_codes, neighbor_radius=CONFIG["local_neighbor_radius"])
else:
    cell_meta = None

# 6. PyG data + patch-grouped loader
print("6. pyg")
data = build_pyg_data(X_svg, conn_sub, patch_row_trans, slide_codes)
patch_to_nodes = build_patch_to_nodes(data.patch_row)
loader = PatchGroupedLoader(
    data, patch_to_nodes, patch_meta["slide_id"],
    num_neighbors=CONFIG["num_neighbors"],
    patches_per_batch=CONFIG["patches_per_batch"],
    max_cells_per_patch=CONFIG["max_cells_per_patch"],
)


### Model

In [ ]:
model = DualAutoencoder(
    n_svg=X_svg.shape[1], hist_dim=hist_emb.shape[1],
    latent_dim=CONFIG["latent_dim"], proj_dim=CONFIG["proj_dim"],
    n_slides=n_slides,
    pool_type=CONFIG["pool_type"], encoder_type=CONFIG["encoder"],
    token_dim=(token_emb.shape[1] if CONFIG["use_tokens"] else None),
)
model.to(device)
hist_emb_t = torch.tensor(hist_emb, dtype=torch.float32)
token_emb_t = torch.tensor(token_emb, dtype=torch.float32) if CONFIG["use_tokens"] else None


### Phase 1a -- pretrain histology (patch) branch

In [ ]:
pretrain_hist_autoencoder(
    model, hist_emb_t, patch_meta,
    epochs=CONFIG["hist_pretrain_epochs"], lr=CONFIG["hist_pretrain_lr"],
    weight_decay=CONFIG["hist_pretrain_weight_decay"],
    val_frac=CONFIG["hist_pretrain_val_frac"], patience=CONFIG["hist_pretrain_patience"],
    device=device,
)


### Phase 1b -- pretrain token (local) branch (separate cell, skipped when `use_tokens=False`)

In [ ]:
if CONFIG["use_tokens"]:
    pretrain_token_autoencoder(
        model, token_emb_t, cell_meta,
        epochs=CONFIG["token_pretrain_epochs"], lr=CONFIG["token_pretrain_lr"],
        weight_decay=CONFIG["token_pretrain_weight_decay"],
        val_frac=CONFIG["token_pretrain_val_frac"], patience=CONFIG["token_pretrain_patience"],
        device=device,
    )


### Phase 2 -- freeze both histology branches

In [ ]:
freeze_hist_encoder(model, freeze_decoder=True)
if CONFIG["use_tokens"]:
    freeze_token_encoder(model, freeze_decoder=True)


### Phase 3 -- joint training (patch-level + cell-level contrastive)

In [ ]:
train(
    model, loader, hist_emb_t, patch_meta,
    n_epochs=CONFIG["n_epochs"], lr=CONFIG["lr"], warmup_epochs=CONFIG["warmup_epochs"],
    max_contrast_weight=CONFIG["max_contrast_weight"],
    weight_reconstruction=CONFIG["weight_reconstruction_by_patch_freq"],
    freeze_hist=True,
    device=device,
    token_emb_all=token_emb_t, cell_meta=cell_meta,
    max_local_contrast_weight=(CONFIG["max_local_contrast_weight"] if CONFIG["use_tokens"] else 0.0),
    local_warmup_epochs=CONFIG["local_warmup_epochs"],
    freeze_token=CONFIG["freeze_token"],
    token_lr_mult=CONFIG["token_lr_mult"],
    patch_anchor_sim_threshold=CONFIG["patch_anchor_sim_threshold"],
    local_anchor_sim_threshold=(CONFIG["local_anchor_sim_threshold"] if CONFIG["use_tokens"] else None),
)


### Eval

In [ ]:
patch_meta_device = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in patch_meta.items()}
eval_results = retrieval_accuracy(model, data, hist_emb_t, patch_meta_device, device=device)

if CONFIG["use_tokens"]:
    cell_meta_device = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in cell_meta.items()}
    token_eval_results = token_retrieval_accuracy(model, data, token_emb_t, cell_meta_device, device=device)


### Save checkpoint + latents

In [ ]:
checkpoint = {
    "model_state_dict": model.state_dict(),
    "epoch": CONFIG["n_epochs"] - 1,
    "n_svg": X_svg.shape[1],
    "hist_dim": hist_emb.shape[1],
    "latent_dim": CONFIG["latent_dim"],
    "proj_dim": CONFIG["proj_dim"],
    "n_slides": n_slides,
    "pool_type": CONFIG["pool_type"],
    "svg_scaler": svg_scaler,
    "hist_scaler": hist_scaler,
    "config": CONFIG,
}
if CONFIG["use_tokens"]:
    checkpoint["token_dim"] = token_emb.shape[1]
    checkpoint["token_scaler"] = token_scaler
torch.save(checkpoint, dir_output / "dual_ae_checkpoint.pt")

with torch.no_grad():
    z_trans_all = encode_full_graph_batched(
        model.trans_enc, data, device,
        num_neighbors=CONFIG["num_neighbors"], batch_size=4096,
    ).cpu().numpy()
    z_hist_all = model.hist_enc(hist_emb_t.to(device)).cpu().numpy()
    if CONFIG["use_tokens"]:
        z_token_all = model.token_enc(token_emb_t.to(device)).cpu().numpy()
z_patch_trans_all = eval_results["z_patch_trans_full"]  # one row per unique patch, pooled

save_kwargs = dict(
    z_trans_cell=z_trans_all,
    z_hist_per_patch=z_hist_all,
    z_patch_trans=z_patch_trans_all,
    patch_row=data.patch_row.numpy(),
    slide_code=data.slide_code.numpy(),
)
if CONFIG["use_tokens"]:
    save_kwargs["z_token_cell"] = z_token_all
np.savez(dir_output / "dual_ae_latents.npz", **save_kwargs)

trans_sub.obsm["X_dae_trans_cell"] = z_trans_all
trans_sub.obsm["X_dae_trans_patch"] = z_patch_trans_all[data.patch_row.numpy()]
trans_sub.obsm["X_dae_hist_patch"] = z_hist_all[data.patch_row.numpy()]
trans_sub.obsm["X_dae_joint_patch"] = np.concatenate(
    [trans_sub.obsm["X_dae_trans_patch"], trans_sub.obsm["X_dae_hist_patch"]], axis=1
)
if CONFIG["use_tokens"]:
    trans_sub.obsm["X_dae_token_cell"] = z_token_all
    # local + regional histology, fused with transcriptomic identity, per cell
    trans_sub.obsm["X_dae_trans_token_hist_concat"] = np.concatenate(
        [trans_sub.obsm["X_dae_trans_cell"], trans_sub.obsm["X_dae_token_cell"], trans_sub.obsm["X_dae_hist_patch"]],
        axis=1,
    )

trans_sub.write_h5ad(dir_output / "trans_with_dae_latents.h5ad")
print("Done. Outputs written to:", dir_output)
